# Pipeline INTERSECT v2 — sửa 2 lỗi đã xác nhận qua review thật (luật sư)

**Bối cảnh:** Review tay 2 câu trả lời thật phát hiện 2 lỗi của `answer_intersect.py` (bản gốc):

| Case | Lỗi | Nguyên nhân kỹ thuật |
|---|---|---|
| Câu 1 (cơ sở ươm tạo) | **Thừa** trích dẫn Nghị quyết 98/2023/QH15 (TP.HCM) — luật sư xác nhận KHÔNG liên quan | Khi intersection rỗng, code gốc **fallback "mù" lấy top-N theo rerank**, không xét nội dung có liên quan ngữ nghĩa hay không |
| Câu 2 (đấu thầu) | **Thiếu** Điều 10 Luật Đấu thầu 22/2023/QH15 — luật sư xác nhận LLM cite đúng nhưng bị bỏ sót | Pool rerank top-8 không chứa chunk đó (Recall của tầng Retrieval/Rerank), nên intersect không có gì để match |

**Cách sửa** (`src/answer_intersect_v2.py`):
1. **Pairing theo vị trí**: ghép "Điều X" với số hiệu văn bản gần nhau nhất trong câu (thay vì 2 set rời rạc như bản gốc).
2. **Bỏ fallback "mù"**: nếu intersection trong pool rỗng nhưng cặp (Điều, văn bản) LLM cite **tồn tại thật trong corpus** (chỉ bị rerank xếp ngoài top-k) → vẫn lấy, qua `corpus_lookup`. Chỉ fallback top-1 (tối thiểu) khi answer không cite được gì hợp lệ — không lấy top-N rác.
3. **Bonus tìm thêm trong lúc viết test**: sửa luôn 1 bug regex (`_DOC_NUM_RE` thiếu `0-9` trong charset → cắt mất hậu tố như "QH14" thành "QH") — bug này **có cả trong bản gốc**.

**Audit dữ liệu** (`src/reference_extractor.py::validate_manifest_consistency`): quét `law_manifest.json` một lần để phát hiện entry có `btc_standard_string` không khớp đúng key của nó — chạy 1 lần, KHÔNG tốn thời gian LLM, không phải bước so sánh pipeline.

> Notebook này chạy TRỰC TIẾP pipeline INTERSECT V2 trên dataset thật của bạn, với số câu bạn tự chọn (ví dụ 20, 100, hoặc toàn bộ). Không chạy lại pipeline cũ để so sánh.
> ⚠️ **Deadline đóng cổng 30/06/2026 23:59 (giờ VN)** — `--llm-answer` tốn ~29s/câu, hãy tự ước lượng thời gian theo số câu bạn chọn trước khi chạy.

In [ ]:
# 1) Cài dependencies (giống bản gốc)
!pip install -q qdrant-client rank_bm25 underthesea sentence-transformers \
    transformers accelerate bitsandbytes python-dotenv tqdm

In [ ]:
# 2) Load secrets Qdrant từ Kaggle
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
    print("✅ Secrets loaded from Kaggle")
except Exception as e:
    print("⚠️ Không load được secrets:", e)

In [ ]:
# 3) Clone / pull code từ GitHub — NHÁNH test1 (giữ nguyên thiết lập của bạn)
#    LƯU Ý: các file mới (answer_intersect_v2.py, sửa fast_retrieval.py +
#    reference_extractor.py) cần được PUSH lên đúng nhánh này trước khi chạy notebook.
import os, sys

GITHUB_USERNAME = "kimmttrung"
REPO_NAME = "legal-rag-vietnam"
BRANCH = "test1"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
WORKING_DIR = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(WORKING_DIR):
    print(f"🔄 Pull nhánh {BRANCH}...")
    !cd {WORKING_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"📥 Clone nhánh {BRANCH}...")
    !cd /kaggle/working && git clone -b {BRANCH} {REPO_URL}

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
print(f"✅ Đồng bộ hoàn tất! (branch={BRANCH})")

In [ ]:
# 4) Vào thư mục repo + kiểm tra file cần thiết (BAO GỒM file mới của v2)
import os, sys

WORKING_DIR = "/kaggle/working/legal-rag-vietnam"
os.chdir(WORKING_DIR)
if sys.path[0] != WORKING_DIR:
    sys.path.insert(0, WORKING_DIR)

print("Working directory:", os.getcwd())
for f in ["fast_retrieval.py", "src/answer_intersect.py", "src/answer_intersect_v2.py",
          "src/reference_extractor.py", "config/settings.py", "data/corpus_clean.json",
          "data/law_manifest.json"]:
    print(("✅ " if os.path.exists(f) else "❌ THIẾU: ") + f)

In [ ]:
# 5) AUDIT MANIFEST (nhanh, không tốn thời gian LLM) — phát hiện lỗi mapping
#    mang tính hệ thống trước khi chạy, KHÔNG phải bước so sánh pipeline.
import json
from src.reference_extractor import load_manifest, validate_manifest_consistency

manifest = load_manifest()
issues = validate_manifest_consistency(manifest)
print(f"\n{'⚠️ CẦN SỬA MANIFEST TRƯỚC KHI CHẠY FULL' if issues else '✅ Manifest sạch'} "
      f"— {len(issues)} entry nghi vấn lệch mapping trên tổng {len(manifest)} entry.")
if issues:
    print("\n5 entry nghi vấn đầu tiên:")
    for m in issues[:5]:
        print("  -", m)

In [ ]:
# 6) Chọn dataset câu hỏi THẬT của bạn + SỐ CÂU BẤT KỲ muốn chạy
#    NUM_QUESTIONS = 0   -> chạy TOÀN BỘ câu hỏi trong file
#    NUM_QUESTIONS = N   -> chỉ chạy N câu ĐẦU TIÊN (vd 20, 100, 500...)
import glob, json

NUM_QUESTIONS = 20   # <-- ĐỔI SỐ NÀY theo ý bạn (0 = chạy hết)

INPUT_FILE = None   # <-- hoặc gán thẳng đường dẫn file câu hỏi của bạn vào đây
if INPUT_FILE is None:
    cands = glob.glob("/kaggle/input/**/*.json", recursive=True)
    print("JSON trong /kaggle/input:")
    for c in cands:
        print("  -", c)
    INPUT_FILE = next((c for c in cands if any(k in c.lower() for k in ["r2ai", "question", "data", "test"])), None) or (cands[0] if cands else None)

assert INPUT_FILE and os.path.exists(INPUT_FILE), "Không tìm thấy INPUT_FILE! Hãy Add Input dataset trên Kaggle hoặc gán thẳng đường dẫn."
NUM_ARG = f"--num-questions {NUM_QUESTIONS}" if NUM_QUESTIONS and NUM_QUESTIONS > 0 else ""
print(f"\n✅ INPUT_FILE = {INPUT_FILE}")
print(f"✅ Số câu sẽ chạy: {'TOÀN BỘ' if not NUM_ARG else NUM_QUESTIONS}")

In [ ]:
# 7) Chạy pipeline INTERSECT V2 (đã sửa 2 lỗi + audit manifest ở trên) trên
#    dataset thật, với số câu bạn đã chọn ở cell trên.
#    pool_k=10 (tăng so với bản gốc pool_k=8) để giảm khả năng "thiếu do rerank
#    xếp ngoài pool" — corpus_lookup vẫn cứu được phần còn lại nếu cần.
print(f"🚀 INTERSECT V2 — {INPUT_FILE} {NUM_ARG}...\n")
!python fast_retrieval.py --input "{INPUT_FILE}" --output /kaggle/working/results.json \
    --llm-answer --intersect-v2 --pool-k 10 --max-select 5 {NUM_ARG}

In [ ]:
# 8) Kiểm tra + đóng gói submission.zip
import json, zipfile, statistics

results = json.load(open("/kaggle/working/results.json", encoding="utf-8"))
na = [len(r["relevant_articles"]) for r in results]
nd = [len(r["relevant_docs"]) for r in results]
has_ans = sum(1 for r in results if r["answer"].strip())
print(f"✅ results.json — {len(results)} bản ghi | answer khác rỗng: {has_ans}")
print("   id mẫu:", [r['id'] for r in results[:5]], "| kiểu:", type(results[0]['id']).__name__)
print(f"   TB điều/câu={statistics.mean(na):.2f} | TB văn bản/câu={statistics.mean(nd):.2f}")

# Cảnh báo nhanh: phát hiện 'doc mồ côi' (relevant_docs không có Điều tương ứng nào)
orphan_count = 0
for r in results:
    doc_codes = {d.split("|")[0] for d in r["relevant_docs"]}
    art_codes = {a.split("|")[0] for a in r["relevant_articles"]}
    if doc_codes - art_codes:
        orphan_count += 1
print(f"   ⚠️ Số câu có 'doc mồ côi' (văn bản không có Điều kèm theo): {orphan_count}")

with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/results.json", arcname="results.json")
print("✅ /kaggle/working/submission.zip (chứa results.json)")

print("\n----- Bản ghi mẫu -----")
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:700])